<div dir="rtl" lang="he" align="right" markdown="1">

# סוכנים: לולאה, כלים, וג'ייסון שבור

ההבדל בין השניים הוא מי בוחר. `workflow` הוא מסלול שכתוב בקוד, והמודל ממלא בו שלבים. `agent` הוא לולאה שבה **המודל בוחר** מה לעשות עכשיו, והקוד רק מריץ את הבחירה ומחזיר לו את התוצאה. זה כל ההבדל, והוא גם ההבדל בין מערכת שאפשר לנפות בה באגים לבין מערכת שצריך למדוד.

הבחירה מגיעה כ-`JSON`. מודלים קטנים שוברים אותו, וזה הפרק על מה בדיוק נשבר.

**שני משתנים ייבדקו כאן**, על אותו כלי בדיוק: כמה ארוך התיאור שלו, וכמה קפדנית הבדיקה שמקבלת את התשובה. לשניהם השפעה גדולה פי שישה ומעלה, ואת שניהם אנחנו בוחרים.

</div>

In [ ]:
import json
from pathlib import Path

from aihe import viz
from aihe.models import asker
from aihe.tools import Parameter, Tool, run_tasks, summarise

HERE = Path("data") if Path("data/tool-tasks.json").exists() else Path("chapters/05-agents/data")
CASSETTES = HERE.parent / "cassettes"
plan = json.loads((HERE / "tool-tasks.json").read_text(encoding="utf-8"))

tools = {}
for spec in plan["tools"]:
    label = "narrow" if len(spec["description"]) < 100 else "wide"
    tools[label] = Tool(spec["name"], spec["description"],
                        tuple(Parameter(**p) for p in spec["parameters"]))
print({k: f"{len(v.parameters)} params, {len(v.description)} chars" for k, v in tools.items()})

<div dir="rtl" lang="he" align="right" markdown="1">

## כלי הוא סכמה, ותו לא

כלי הוא שם, תיאור, ורשימת פרמטרים. זה מה שהמודל רואה, וזה כל מה שיש לו כדי להחליט.

נסתכל על אותה יכולת בדיוק, מתוארת בשתי דרכים. הראשונה בשורה אחת ושלושה פרמטרים. השנייה כמו שקוד אמיתי מתאר כלי: שש שורות וחמישה פרמטרים. אין בשנייה שום דבר לא סביר — היא פשוט ארוכה יותר.

</div>

In [ ]:
print("NARROW ---")
print(tools["narrow"].prompt_hint())
print("\nWIDE ---")
print(tools["wide"].prompt_hint()[:300], "...")

<div dir="rtl" lang="he" align="right" markdown="1">

## הלולאה

נבקש מהמודל לקרוא לכלי, ננסה לקרוא את ה-`JSON` שחזר, ונבדוק אותו מול הסכמה. אם הוא נפסל — נחזיר למודל את הודעת השגיאה ונבקש שוב, עד מספר ניסיונות חסום.

החסימה חשובה. לולאת תיקון בלי גבול מול מודל שלא מסוגל לעמוד בסכמה היא חשבון אינסופי.

</div>

In [ ]:
ask_model = asker(cassettes=CASSETTES, model="Llama-3.2-3B-Instruct-Q4_K_M",
                  temperature=0.0, max_tokens=64, seed=7)

# run_tasks lives in aihe/tools.py, not here: a notebook cell may not define logic, and the
# recorder calls the same function, so the prompts are byte-identical and the cassettes match.
strict_narrow = run_tasks(tools["narrow"], plan["tasks"], ask_model,
                          max_attempts=plan["max_attempts"], null_is_absent=False)
print(f"narrow schema, strict validator: {summarise(strict_narrow)['first_try']:.0%} first try")

<div dir="rtl" lang="he" align="right" markdown="1">

## מה בעצם נשבר

לפני שמאשימים את המודל, כדאי להסתכל על מה הוא באמת כתב. התא הבא מדפיס את הניסיון הראשון שנפסל, בדיוק כמו שהגיע.

</div>

In [ ]:
for result in strict_narrow:
    first = result.attempts[0]
    if not first.valid:
        print("the model wrote:", first.raw.strip()[:110])
        print("we rejected it :", first.errors)
        break

<div dir="rtl" lang="he" align="right" markdown="1">

## הבדיקה היא שטעתה

תסתכלו שוב על מה שנפסל. המודל כתב `null` לשדה שהסכמה עצמה מגדירה כאופציונלי. במילים אחרות הוא אמר "את זה אני לא רוצה" — וזו תשובה נכונה לגמרי.

הבדיקה שלנו דחתה אותה. זו לא טעות של המודל אלא החלטה שלנו, והיא עולה כסף: מתוך כל הכשלים בניסיון הראשון, בשני הכלים יחד, `89%` הם בדיוק המקרה הזה.

נריץ את אותו דבר בדיוק עם בדיקה שקוראת `null` כ"לא נמסר", עבור שדות אופציונליים בלבד. שדה חובה שקיבל `null` נשאר שגיאה, כי זה סירוב לעשות את העבודה.

</div>

In [ ]:
rows = {}
for label, tool in tools.items():
    for lenient in (False, True):
        results = run_tasks(tool, plan["tasks"], ask_model,
                            max_attempts=plan["max_attempts"], null_is_absent=lenient)
        rows[(label, "lenient" if lenient else "strict")] = summarise(results)

print(f"{'schema':10s} {'validator':10s} {'first try':>10s} {'eventually':>11s} {'mean tries':>11s}")
for (label, mode), s in rows.items():
    print(f"{label:10s} {mode:10s} {s['first_try']:10.0%} {s['eventually']:11.0%} "
          f"{s['mean_tries']:11.2f}")

<div dir="rtl" lang="he" align="right" markdown="1">

## שני משתנים, שניהם בידיים שלנו

הטבלה למעלה מחזיקה שני ממצאים נפרדים, ושניהם גדולים.

**אורך התיאור.** אותו כלי, אותן משימות, רק תיאור ארוך יותר: `first try` ירד מ-`75%` ל-`12%` תחת הבדיקה הקפדנית. תיאור ארוך הוא לא תיעוד טוב — הוא רעש שהמודל צריך לעבור דרכו.

**קפדנות הבדיקה.** אותה סכמה רחבה בדיוק, עם בדיקה שמקבלת `null` כ"לא נמסר": `12%` הפכו ל-`88%`.

שימו לב שגם לולאת התיקון עשתה את שלה — היא הביאה את הכלי הרחב ל-`100%` הצלחה בסוף גם כשההתחלה הייתה `12%`. אבל היא עשתה את זה בקריאות נוספות, וכל קריאה עולה זמן וכסף.

</div>

In [ ]:
viz.climb({"wide + strict": rows[("wide", "strict")]["first_try"],
           "narrow + strict": rows[("narrow", "strict")]["first_try"],
           "wide + lenient": rows[("wide", "lenient")]["first_try"],
           "narrow + lenient": rows[("narrow", "lenient")]["first_try"]},
          label="valid on the first try",
          title="the same tool and the same tasks - only our choices change");

<div dir="rtl" lang="he" align="right" markdown="1">

## ולבסוף, לחבר את זה למה שבנינו

הכלי בפרק הזה הוא כלי חיפוש, וכבר בנינו אחד בפרק 01. סוכן שמחפש במסמכים הוא בדיוק הלולאה הזו סביב ה-`retriever` שכתבנו שם: המודל בוחר שאילתה, הקוד מריץ את החיפוש, התוצאה חוזרת, והמודל מחליט אם זה מספיק.

וזה גם מסגר את כל הקורס. אם האחזור מחזיר את המסמך הלא נכון, הסוכן החכם ביותר בעולם יענה תשובה מוטעית בביטחון מלא — ולכן ארבעת הפרקים הראשונים באים לפני הפרק הזה ולא אחריו.

</div>

In [ ]:
best = rows[("narrow", "lenient")]
worst = rows[("wide", "strict")]
print(f"worst combination: {worst['first_try']:.0%} first try, {worst['mean_tries']:.2f} calls per task")
print(f"best combination : {best['first_try']:.0%} first try, {best['mean_tries']:.2f} calls per task")
print(f"same model, same tasks, {worst['mean_tries'] / best['mean_tries']:.1f}x the calls")